# Task
Design a movie ticket booking system similar to BookMyShow that allows users to browse movies, select theaters and showtimes, book tickets, and manage reservations.

### Clarifying Questions

Q: Can user search for movies by keywords or names of actors?

Users can search for movies by title only — simple substring matching on the movie title. Searching by actor names or other keywords is out of scope.

Q: Is there a movie detail page that shows description and theatres?

Yes, think of it as a backend system — so there's no literal "page," but the system should support fetching movie details (like description) and retrieving the list of theaters showing a given movie.

Q: Can user select a theatre by location and system returns the movies showing on that theatre?

Yes, that's supported — users can browse movies showing at a specific theater. Location-based search (e.g., by city or coordinates) is out of scope; theaters are selected directly rather than filtered by location.

Q: So system returns a complete list of theatres and user picks one from there?

Yes, exactly — the system supports retrieving a list of all theaters, and the user selects one from that list.

Q: Theatres having multiple screens means they can show different movies in parallel?

Yes, exactly — a theater can have multiple screens, and each screen can show a different movie at any given time (or even the same movie at different showtimes).

Q: When user book tickets, how is payment done?

Payment processing is out of scope for this system — you can assume payment always succeeds. Your design doesn't need to handle any payment logic.

Q: What does manage reservation need to cover?

For this system, managing reservations covers two operations: viewing a reservation's details by confirmation ID, and canceling a reservation by confirmation ID.

Q: what does reservation details have? And happens when a reservation is cancelled?

A reservation's details include: the confirmation ID, the movie and showtime, the theater and screen, the list of booked seats, and the user who made the booking.
When a reservation is cancelled, the seats that were held under that reservation become available again for other users to book.

Q: Different theatres can have different seat maps, is that something I need to consider?

No, variable seat layouts are out of scope. All screens across all theaters share the same seat layout: rows A–Z with seats numbered 0–20.

Q: Does the system need to return a complete list of all movies currently showing?

Yes, the system should support retrieving a list of all movies that are currently showing (i.e., have at least one active showtime).

Q: Is movie and showtime management out of scope?

Yes, movie and showtime management — such as adding movies, creating theaters/screens, and scheduling showtimes — is in scope as part of the backend system. You'll need to support those operations alongside the user-facing booking flows.

Q: What happens if admin tries to delete a movie or modify showtimes when there are seats already reserved?

Good question. Keep it simple — deleting a movie or modifying a showtime that has existing reservations is not allowed. The system should reject such operations if there are active reservations tied to that movie or showtime.

Q: What are the things that are out of scope?

Here's a summary of what's out of scope for this system:
- Payment processing — assume payment always succeeds
- Variable seat layouts — all screens use the same fixed layout (rows A–Z, seats 0–20)
- Location-based theater search — no filtering theaters by city or coordinates
- Searching movies by actor names or keywords — title substring matching only
- Rescheduling reservations — only viewing and canceling are supported
- UI rendering — backend logic only
- Deleting/modifying showtimes or movies with active reservations — the system simply rejects such operations

Q: Should exceptions be raised for all errors?

Yes, use exceptions (or a equivalent error-signaling mechanism) for invalid operations — for example, trying to book an already-taken seat, canceling a non-existent reservation, or attempting to delete a movie with active reservations. How you structure or name those exceptions is up to you.

Q: What is the scale or load for this system?

For this interview, treat it as a single-machine, in-memory system — no need to design for distributed scale or high availability. Focus on correctness of the design, particularly around concurrent seat booking.

Q: users (admins) who can modify movies, showtimes and theatres have different privileges that regular users, how is that handle?

For this system, you can keep it simple — there's no authentication or role-based access control to implement. Assume that admin operations (like adding movies, creating theaters, scheduling showtimes) and user operations (like booking and canceling tickets) are just different parts of the API, and any caller invoking them is authorized to do so. You don't need to model users, admins, or privileges as part of this design.

Q: Does the system need to keep track of show duration? When user adds a new showtime for a screen, should the system check for any conflict (that is a movie showing on the screen and has not ended)?

Yes, the system should store the duration of a movie. And yes, when adding a new showtime to a screen, the system should check for scheduling conflicts — if an existing showtime on that screen hasn't ended yet (based on start time + duration), the new showtime should be rejected.

Q: Should the system remove shows that already expired?

No, the system does not need to automatically clean up or remove expired showtimes. You can assume they persist in the system indefinitely — just focus on correctness of the booking and scheduling logic.


# 1. Requirements

- System should list all available movies
- System should list all available theatres
- System should return movie details including title, description, duration, all theatres, screens and showtimes for the movie.
- System should return all movies, and their showtimes and screens for a theatre.
- User can search movies by their titles (supporting simple substring matching)
- User can make a reservation by selecting a movie, theatre, screen, showtime and seats (all theaters share the same seat layout: rows A–Z with seats numbered 0–20)
- System needs to make sure no overbooking, that is user cannot select seats that are already reserved.
- User can view reservation by confirmation id. System should return movie, theatre, screen, showtime, seats reserved, and user.
- User can cancel a reservation by providing its confirmation id. Once cancelled, the seats will return to the pool and available for booking.
- User(admin) can create, modify, delete theatres and screens
- User(admin) can create, modify, delete movies and their showtimes
- System should throw exceptions for invalid operations that includes:
    - trying to retrieve a non-exist movie or theatre
    - making reservation with invalid entries (invalid movie, theatre, screen, showtime or seats)
    - trying to book an already-taken seat
    - cancelling a non-exist reservation
    - delete a movie, screen, theatre or change showtime with active reservations. 
    - name conflict when creating or modifying a theatre, screen or movie
    - trying to delete a non-exist movie, screen, theatre or showtime.  
    - scheduling conflicts — if an existing showtime on that screen hasn't ended yet (based on start time + duration)
- System runs on a single machine and in-memory. 

Out of Scope:
- Payment processing — assume payment always succeeds
- Variable seat layouts — all screens use the same fixed layout (rows A–Z, seats 0–20)
- Location-based theater search — no filtering theaters by city or coordinates
- Searching movies by actor names or keywords — title substring matching only
- Rescheduling reservations — only viewing and canceling are supported
- UI rendering — backend logic only
- Deleting/modifying showtimes or movies with active reservations — the system simply rejects such operations
- Authentication or role-based access control 


### Feedback

What went well

You covered all the core operations thoroughly — searching movies by title, browsing by theater, viewing seat availability, booking multiple seats, and managing reservations (view and cancel). That's a complete picture of the user-facing capabilities. 
Your error handling section is impressively thorough. You covered all the important cases like double-booking, invalid confirmation IDs, scheduling conflicts, and deletion constraints — this kind of thinking upfront will save you a lot of pain during implementation. 
You made great use of your clarifying questions — the requirements are tightly consistent with the answers you received, including the fixed seat layout, no authentication, in-memory single-machine scope, and conflict detection when scheduling new showtimes. 

Minor suggestions

One small thing worth making explicit: the concurrent booking requirement — when two users try to book the same seat simultaneously, exactly one should succeed and the other should get an error. You touch on this implicitly with the 'no overbooking' rule, but calling it out as a concurrency guarantee (rather than just a validation check) would signal to an interviewer that you're thinking about thread safety from the start.

### Sample Answer

Requirements:
1. Users can search for movies by title
2. Users can browse movies playing at a given theater
3. Theaters have multiple screens; all screens share the same seat layout (rows A-Z, seats 0-20)
4. Users can view available seats for a showtime and select specific ones
5. Users can book multiple seats in a single reservation; booking returns a confirmation ID
6. Concurrent booking of the same seat: exactly one succeeds
7. Users can cancel a reservation by confirmation ID, releasing the seats

Out of Scope:
- Payment processing (assume payment always succeeds)
- Variable seat layouts or seat types (all seats identical)
- Rescheduling (cancel and rebook instead)
- UI / rendering

# 2. Entities and Relationships

### Entities
- BookingSystem
- Movie
- Theatre
- Screen
- Show
- Reservation
- User

##### Feedback
What went well

You nailed the core entities — Movie, Theatre, Screen, Show, and Reservation are exactly the right building blocks for this system. 
Including BookingSystem as an orchestrator is a great call, since you'll need something to own the theaters and handle cross-cutting queries like search. 
User is a solid addition that will come in handy when storing reservation details and tying bookings back to a person. 

##### Sample Answer
- BookingSystem
- Theater
- Showtime
- Movie
- Reservation
- Seat (string identifier)
- Screen (string label)

### Relationships
BookingSystem is the orchestrator class. It owns complete lists of movies, theatres, reservations and users. It exposes API for querying movies by title, showing movie details, theatre details, and listing all available movies and theatres. It also owns making reservations (booking seats) and reservation management, as well as theatres, movies and shows management. The BookingSystem class coordinates operation among entities. 

Movie is a data class with properties like id, title, description and duration. 

Theatre is data class that has id, name and location and owns lists of screens and shows.

Show is data class with properties like id, movie id, start time, end time, screen id, seat map (map for seat availability).

Reservation is data class with properties like confirmation id, user id, theatre id, show id and list of seat ids.

User is data class with properties like user id and username.   

##### Feedback
What went well

You clearly identified BookingSystem as the orchestrator — explaining that it coordinates operations across movies, theatres, reservations, and users is exactly the right mental model. 
Your ownership chain is well laid out: Theatre owns Screens and Shows, Show owns the seat map and ties together a movie and a screen, and Reservation ties back to the show with a list of seat IDs. That hierarchy makes a lot of sense. 

Minor suggestions

One small thing to consider: since Reservation needs to route back to its Show during cancellation (to free up seats), you might want to think about whether Reservation holds a direct reference to Show rather than just storing a show ID — it can simplify the lookup logic inside BookingSystem.

##### Sample Answer

"BookingSystem is the orchestrator and entry point. It owns all Theaters. Each Theater owns a list of Showtimes. Showtime is the bookable unit - it references its Theater and Movie, and owns all Reservations for that screening. Reservation has a back-reference to its Showtime so cancellations can be routed to the right place. The reservations list on Showtime is the single source of truth for seat state - if a seat appears in any reservation, it's booked."


# 3. Class Design

The BookingSystem class is the orchestrator and entry point for the system. It owns list of theatres and list of movies. It also owns mappings between theatres and movies, so that it can return movies for a particular theatre or theatres for a movie in O(1) lookup time. The class is responsible for theatre, movie managements, and making reservations and reservation management. 

The Theatre class owns a list of screens and shows. It also owns a mapping between movie and shows, so that it can return the list of shows for a movie in O(1) lookup time.

The Show class owns seat map which is a dictionary with seat Id as key and reservation Id as value. For available seats, the value is none. When they are being reserved, their values will be assigned with reservation Id. The class also has getLock() which will be used for handling concurrency.   

The Movie, Reservation and User classes are just data classes with getters and setters.

```
class BookingSystem:
    - theatres: map<theatre_id, Theatre>
    - movies: map<movie_id, Movie>
    - reservations: map<reservation_id, Reservation>
    - users: map<user_id, User>
    - _movie_theatre_mapping: Map<movie_id, list<Theatre>>
    - _theatre_movie_mapping: Map<theatre_id, list<Movie>>
    + BookingSystem() -> BookingSystem
    + addTheatre(Theatre)
    + addMovie(Movie)
    + deleteTheatre(Theatre)
    + deleteMovie(Movie)
    + getAllMovies() -> list<Movie>
    + getAllTheatres() -> list<Theatre>
    + getTheatresByMovie(Movie) -> list<Theatre>
    + getMoviesByTheatre(Theatre) -> list<Movie>
    + searchMoviesByTitle(search_string) -> list<Movie>
    + makeReservation(User, Show, seats: list<string>) -> string
    + viewReservation(reservation_id) -> Reservation
    + cancelReservation(reservation_id)

class Theatres:
    - _id : string
    - _name : string
    - _location : string
    - _screens: Set<string>
    - _shows: list<show>
    - _movie_show_mapping: map<movie_id, list<Show>>
    + Theatre(name, location) -> Theatre
    + getId() -> string
    + getName() -> string
    + setName(name)
    + getLocation() -> string
    + setLocation(string)
    + addScreen(screen_name)
    + removeScreen(screen_name)
    + addShow(Movie, start_time, end_time, screen_name)
    + removeShow(Show)
    + getShowsByMovie(Movie) -> list<Show>


class Show:
    - _lock: Lock
    - _movie: Movie
    - _start_time : datetime
    - _end_time : datetime 
    - _screen_name : string
    - _seat_map : map<seat_id, reservation_id?>
    - _theatre : Theatre
    + Show(Movie, start_time, end_time, screen_name, seat_map) -> Show
    + getAvailableSeats() -> list<string>
    + hasReservedSeats() -> bool
    + isSeatAvailable(seat_id) -> bool
    + reserveSeats(list<string>, reservation_id)
    + cancelSeats(list<string>)
    + getShowStarttime() -> datetime
    + getShowEndTime() -> datetime
    + getTheatre() -> Theatre
    + getLock() -> Lock

class Movie:
    - _id : string
    - _title : string
    - _description: string
    - _duration_in_min: int
    + Movie(title, description, duration_in_min) -> Movie
    + getId() -> string
    + getTitle() -> string
    + setTitle(title)
    + getDescription() -> string
    + setDescription(description)
    + getDurationInMinutes() -> int
    + setDurationInMinutes(duration)


class Reservation:
    - _id : string
    - _user : User
    - _show : Show
    - _seats : list<string>
    + Reservation(reservation_id, user, show, seats) -> Reservation
    + getShow() -> Show
    + getSeats() -> list<string>
    + getUser() -> User


class User:
    - _id : strinng
    - _username : string
    + User(id, username) -> User
    + getId() -> string
    + getUsername() -> string
    + setUsername(username)
```

### Feedback

What went well

Your Show class is the star of this design — it owns the _seat_map as the single source of truth for seat availability, exposes isSeatAvailable(), reserveSeats(), and cancelSeats(), and even includes a _lock with getLock() showing you're already thinking about the concurrency problem. That's exactly the right instinct. 

Your Reservation holds a back-reference to _show, which is the key insight that makes cancelReservation() work efficiently — BookingSystem can look up the reservation and immediately route the cancel to the right Show without any searching. 

The dual mappings you defined — _movie_theatre_mapping in BookingSystem and _movie_show_mapping in Theatre — show you thought carefully about query performance, making O(1) lookups possible for 'theatres showing this movie' and 'shows for this movie at this theatre'. 

### Sample Answer

"This design separates concerns cleanly: BookingSystem orchestrates cross-theater queries and routes booking/cancellation. Showtime owns both seat state and reservations - the reservations list is the single source of truth. Reservation has a back-reference to Showtime for efficient cancellation routing. Seat and Screen are simple strings since they don't need behavior."

```
class BookingSystem:
    - theaters: List<Theater>

    + searchMovies(title: string) -> List<Showtime>
    + getShowtimesAtTheater(theater: Theater) -> List<Showtime>
    + book(showtimeId: string, seatIds: List<string>) -> Reservation
    + cancelReservation(confirmationId: string)

class Theater:
    - id: string
    - name: string
    - showtimes: List<Showtime>

    + getShowtimes() -> List<Showtime>
    + getShowtimesForMovie(movie: Movie) -> List<Showtime>

class Showtime:
    - id: string
    - theater: Theater
    - datetime: DateTime
    - screenLabel: string
    - movie: Movie
    - reservations: List<Reservation>

    + getId() -> string
    + getTheater() -> Theater
    + getDatetime() -> DateTime
    + getMovie() -> Movie
    + isAvailable(seatId: string) -> boolean
    + getAvailableSeats() -> List<string>
    + book(reservation: Reservation)
    + cancel(reservation: Reservation)

class Movie:
    - id: string
    - title: string

    + getTitle() -> string
    + getId() -> string

class Reservation:
    - confirmationId: string
    - showtime: Showtime
    - seatIds: List<string>

    + getConfirmationId() -> string
    + getSeatIds() -> List<string>
    + getShowtime() -> Showtime

Constants:
    SEAT_LAYOUT: rows A-Z, seats 0-20 (546 seats per showtime)
```

# 4. Implementation

Implement reserveSeats(list<string>, reservation_id) on Show. This is the core booking logic — walk me through how you'd use _lock to safely handle the case where two users try to book the same seat at the same time, and what happens if one of the requested seats is already taken mid-way through the list.

### My Answer

When there are 2 users trying to reserve the same seat, only one of them will be able to get the _lock inside reserveSeats(). Call from the other user will have to wait until the lock is released. The lock will not be released until the whole operation is completed, since everything is inside a block. 

Notice I have first checked the availability for all seats before updating the seat map.  This is because if one of the seats is not available, the whole transaction should not go through.
 
When it is the other user's turn, the isSeatAvailable() will return False and thus resulting in error.

### Feedback

What went well

Your two-pass approach is exactly right — checking isSeatAvailable() for all seats first, then writing, gives you the all-or-nothing atomicity the problem demands. 
You correctly reasoned about the lock: holding _lock for the entire check-then-write block is what prevents two concurrent bookings from both passing the availability check for the same seat. 
Minor suggestions

Consider guarding against a seat_id that doesn't exist in _seat_map (invalid seat), since self._seat_map[seat_id] would throw a KeyError rather than a clean validation error. 

In [ ]:
class Show:
    def isSeatAvailable(self, seat_id) -> bool:
        return self._seat_map[seat_id] == None
    
    def reserveSeats(self, seats: list[str], reservation_id: str):
        with self._lock:
            for seat_id in seats:
                if not self.isSeatAvailable(seat_id):
                    raise SeatNotAvailableError(seat_id)
                
            for seat_id in seats:
                self._seat_map[seat_id] = reservation_id

Implement makeReservation(User, Show, seats: list<string>) on BookingSystem. Walk me through the full orchestration — what validations do you perform, how do you create the Reservation, how do you coordinate with Show.reserveSeats(), and what does the method return?

### My Answer

The method will check to make sure all inputs are not empty and they are valid. Also, need to make sure the show has not expired.  When everything is ready, an unique reservation Id is created. It will be passed to Show.reserveSeats(). reserveSeats() will raise error if seats are no longer available. Otherwise, a reservation will be created and map to the reservation_id. Finally, the reservation will be returned to the caller.   

### Feedback

What went well

Your validation ordering is clean — checking user, theatre, show, expiry, and empty seats before touching any state is exactly the right defensive approach. 
You correctly delegated the atomic seat check to Show.reserveSeats() under a lock, doing the all-or-nothing availability check before mutating _seat_map, which solves the concurrent booking requirement well. 

Generating the reservation_id first and passing it into reserveSeats() before creating the Reservation gives you a clean ordering that ties seats to the reservation. 
What needs improvement

You never validate that the requested seat_ids actually exist in the layout — an invalid seat like "Z99" would hit self._seat_map[seat_id] in isSeatAvailable() and raise a KeyError instead of a meaningful InvalidSeatError. 
Minor suggestions

Consider what happens if show.reserveSeats() succeeds but registering the reservation fails — since the seats are already claimed, you'd want to be sure that path can't leave orphaned seat holds


In [ ]:
def makeReservation(self, user: User, show: Show, seats: list[str]) -> Reservation:
        if not user or user.getId() not in self._users:
            raise InvalidUserError
        
        theatre = show.getTheatre()
        if not theatre or theatre.getId() not in self._theatres:
            raise InvalidTheatreError

        if not show or not threater.hasShow(show):
            raise InvalidShowError

        if show.hasExpired():
            raise ExpiredShowError

        if not seats:
            raise NoSeatSelectedError
         
        reservation_id = uuid.uuid4()
        show.reserveSeats(seats, reservation_id)

        reservation = Reservation(reservation_id, user, show, seats)
        self._reservations[reservation_id] = reservation
        return reservation

Implement cancelReservation(reservation_id) on BookingSystem. Show me how you look up the reservation, free the seats, and clean up state — and whether you need to acquire any locks here.

### My Answer
The cancelReservation() will check if reservation id corresponds to an existing reservation and the show is not yet expired. It then calls the show.cancelSeats() to free up the seats. Inside cancelSeats(), a lock is used. Finally, the reservation is removed from _reservations list after cancelSeats() succeeded.

### Feedback
What went well

Your lookup-and-validate flow is clean: you check the reservation exists, free the seats via show.cancelSeats(), then remove from _reservations only after the cancel succeeds. 

You correctly delegated locking to cancelSeats() on the Show, which keeps the seat-map mutation synchronized where the shared state actually lives. 

What needs improvement

Blocking cancellation with ExpiredShowError when show.hasExpired() may prevent legitimate cleanup — think about whether a user should always be able to cancel, even for a past show. 

Minor suggestions

In cancelSeats() you blindly null out seats; consider verifying each seat's current reservation_id matches before freeing, so a stale call can't accidentally release seats owned by someone else. 

In [ ]:
    def cancelReservation(self, reservation_id):
        if reservation_id not in self._reservations:
            raise InvalidReservationError
        
        reservation = self._reservations[reservation_id]
        show = reservation.getShow()
        if show.hasExpired():
            raise ExpiredShowError

        show.cancelSeats(reservation.getSeats())
        del self._reservations[reservation_id]

    def cancelSeats(self, seats: list[str]):
        with self._lock:
            for seat_id in seats:
                if seat_id in self._seat_map:
                    self._seat_map[seat_id] = None
    

# 5. Extensibility

Right now, makeReservation() immediately claims seats. But in real booking flows, users often spend several minutes on a payment page before confirming — during which those seats should be held but not permanently booked. How would you introduce a temporary seat hold into your current design? Think about what state would need to be added to Show and its _seat_map, how hold expiration would be tracked and enforced, and what would happen to reserveSeats() and cancelSeats() if a hold expires mid-transaction.


In [ ]:

import threading
import uuid

class BookingSystem:
    # - theatres: map<theatre_id, Theatre>
    # - movies: map<movie_id, Movie>
    # - reservations: map<reservation_id, Reservation>
    # - users: map<user_id, User>
    # - _movie_theatre_mapping: Map<movie_id, list<Theatre>>
    # - _theatre_movie_mapping: Map<theatre_id, list<Movie>>
    def __init__(self):
        pass

    # + addTheatre(Theatre)
    # + addMovie(Movie)
    # + deleteTheatre(Theatre)
    # + deleteMovie(Movie)
    # + getAllMovies() -> list<Movie>
    # + getAllTheatres() -> list<Theatre>
    # + getTheatresByMovie(Movie) -> list<Theatre>
    # + getMoviesByTheatre(Theatre) -> list<Movie>
    # + searchMoviesByTitle(search_string) -> list<Movie>
    def makeReservation(self, user: User, show: Show, seats: list[str]) -> Reservation:
        if not user or user.getId() not in self._users:
            raise InvalidUserError
        
        theatre = show.getTheatre()
        if not theatre or theatre.getId() not in self._theatres:
            raise InvalidTheatreError

        if not show or not threater.hasShow(show):
            raise InvalidShowError

        if show.hasExpired():
            raise ExpiredShowError

        if not seats:
            raise NoSeatSelectedError
         
        reservation_id = uuid.uuid4()
        if show.reserveSeats(seats, reservation_id):
            reservation = Reservation(reservation_id, user, show, seats)
            self._reservations[reservation_id] = reservation
            return reservation

        raise UnableToMakeReservationError

    
    # + viewReservation(reservation_id) -> Reservation
    def cancelReservation(self, reservation_id):
        if reservation_id not in self._reservations:
            raise InvalidReservationError
        
        reservation = self._reservations[reservation_id]
        show = reservation.getShow()
        if show.hasExpired():
            raise ExpiredShowError

        show.cancelSeats(reservation.getSeats())
        del self._reservations[reservation_id]



class SeatNotAvailableError(Exception):
    def __init__(self, seat_id: str):
        super().__init__(f"Seat '{seat_id}' is not available.")

class Show:
    def __init__(self, movie, start_time, end_time, screen_name, theatre, seat_map: dict):
        self._movie = movie
        self._start_time = start_time
        self._end_time = end_time
        self._screen_name = screen_name
        self._seat_map = seat_map
        self._theatre = theatre
        self._lock = threading.Lock()

    def getAvailableSeats(self) -> list[str]:
        return [res_id for res_id in self._seat_map.values() if res_id == None]
    
    def isSeatAvailable(self, seat_id) -> bool:
        if seat_id not in self._seat_map:
            raise InvalidSeatError
        return self._seat_map[seat_id] == None
    
    def reserveSeats(self, seats: list[str], reservation_id: str) -> bool:
        with self._lock:
            for seat_id in seats:
                if not self.isSeatAvailable(seat_id):
                    raise SeatNotAvailableError(seat_id)
                
            for seat_id in seats:
                self._seat_map[seat_id] = reservation_id

        return True

    # + hasReservedSeats() -> bool                    
    def cancelSeats(self, seats: list[str], reservation_id: str):
        with self._lock:
            for seat_id in seats:
                if seat_id in self._seat_map and self._seat_map[seat_id] == reservation_id:
                    self._seat_map[seat_id] = None

    # + getShowStarttime() -> datetime
    # + getShowEndTime() -> datetime
    # + getTheatre() -> Theatre
    # + getLock() -> Lock